<a href="https://colab.research.google.com/github/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised/blob/main/erps_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Trial-level ERP analysis**

This notebook performs the ERP-analysis stage of the revised trial-level reanalysis of the Toffolo et al. (2022) N400 dataset.

This notebook links the predictors from stimuli_analysis.ipynb to individual EEG trials and performs both participant-level and group-level statistical analyses.

The pipeline first exports the retained ERP data into a long-format representation, preserving the correspondence between each retained EEG epoch and its original experimental event. Participant-specific design matrices are then constructed by matching every retained trial with its linguistic predictors. These design matrices are subsequently used to estimate participant-level mass-univariate general linear models across electrodes and time points. Finally, component-level linear mixed-effects models are fitted to the Recognition Potential (RP), N400 and Late Positive Component (LPC) in order to quantify the relationship between linguistic predictors and ERP amplitudes while accounting for repeated observations across participants and stimuli.

**Export retained ERP trials**

This section converts the processed ERP derivatives into a trial-level long-format dataset suitable for statistical analysis.

Each retained EEG epoch is matched to its original experimental event using the EEGLAB `urevent` information and target-word onset. The corresponding stimulus identifier (`stim_key`) is recovered and combined with the EEG amplitudes, electrode metadata and temporal information.

The resulting dataset preserves every retained EEG observation and provides the common input for the subsequent ERP analyses. Participant-specific retained-trial lookup tables are also generated for later construction of the first-level design matrices.

In [1]:
%cd /content

!rm -rf Semantically_Incongruent_or_Congruent_Eggplants_revised
!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised

/content
Cloning into 'Semantically_Incongruent_or_Congruent_Eggplants_revised'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 400 (delta 79), reused 47 (delta 29), pack-reused 280 (from 1)
Receiving objects: 100% (400/400), 94.78 MiB | 27.82 MiB/s, done.
Resolving deltas: 100% (227/227), done.
/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


In [2]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.4 MB/s eta 0:00:00


In [3]:
!unzip language_outputs.zip

Archive:  language_outputs.zip
   creating: language_outputs/
  inflating: language_outputs/ALL_language_metrics.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_vif.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_correlations.tsv  
  inflating: language_outputs/ALL_language_metrics_GLM.tsv  


**Note**: Due to their size, the N400 ERP derivatives are stored on Google Drive and mounted during notebook execution.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!unzip -o erps.zip

Archive:  erps.zip
  inflating: erps/task-N400Stimset_erp-CP_trialrej.json  
  inflating: erps/task-N400Stimset_erp-GA_filter.json  
  inflating: erps/task-N400Stimset_erp-GA_trialrej.json  
  inflating: erps/task-N400Stimset_erp-LD_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Order_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Time_trialrej.json  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Time.mat  
  inflating: erps/sub-01/sub

In [5]:
%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised/

/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


# **ERP export analyses**

The ERP derivatives are already split into five analysis schemes: CP, GA, LD, Order, and Time. Each analysis has its own `.mat` file and matching `*_trialrej.tsv` file for each subject. The export script reads those existing files and exports the retained ERP trials to TSV.

Each output keeps the subject, analysis name, condition number, condition label, retained-trial index, original EEGLAB `urevent_index`, channel, timepoint, and amplitude. If the retained-trial count in the `.mat` file differs from the count reported in the `*_trialrej.tsv`, the script uses the `.mat` count because the `.mat` file contains the actual ERP data.

### CP

Exports the Cloze Probability analysis. Trials are grouped by congruency and predictability band, so this output keeps whether each sentence ending was congruent/incongruent and how predictable it was.

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses CP \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_CP"

Found 20 ERP MAT files.
Using ERP root: /content/drive/MyDrive/N400/erps
Analyses: CP
Channels: ALL

Processing sub-01_task-N400Stimset_erp-CP.mat
  Using sub-01_task-N400Stimset_erp-CP_trialrej.tsv
  Channel filter: ALL channels
  Condition 1: Congruent: Cloze Probability ≥ 96 % - 54 before rejection, 29 reported retained, 29 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 2: Congruent: 96 % > Cloze Probability ≥ 90 % - 56 before rejection, 40 reported retained, 40 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 3: Congruent: 90 % > Cloze Probability ≥ 80 % - 47 before rejection, 30 reported retained, 30 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 4: Congruent: Cloze Probability < 80 % - 43 before rejection, 26 reported retained, 26 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 5: Incongruent: Cloze Probability ≥ 96 % - 55 before rejection, 33 reported retained, 33 MAT retained, 615 timep

In [6]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses GA \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_GA"

Found 24 ERP MAT files.
Using ERP root: /content/drive/MyDrive/N400/erps
Analyses: GA
Channels: ALL

Processing sub-01_task-N400Stimset_erp-GA.mat
  Using sub-01_task-N400Stimset_erp-GA_trialrej.tsv
  Channel filter: ALL channels
  Condition 1: Congruent - 200 before rejection, 125 reported retained, 125 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 2: Incongruent - 200 before rejection, 113 reported retained, 113 MAT retained, 615 timepoints, 128 of 128 channels exported
Saved ERP long file: /content/drive/MyDrive/N400/eeg_outputs_GA/sub-01_erp-GA_long.tsv
Saved retained-trial lookup: /content/drive/MyDrive/N400/eeg_outputs_GA/sub-01_erp-GA_trial_lookup.tsv

Processing sub-02_task-N400Stimset_erp-GA.mat
  Using sub-02_task-N400Stimset_erp-GA_trialrej.tsv
  Channel filter: ALL channels
  Condition 1: Congruent - 200 before rejection, 186 reported retained, 187 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 2: Incongruent - 200 before rej

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses LD \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_LD"

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses Order \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_Order"

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses Time \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_Time"